In [1]:
import spacy
from spacy.matcher import PhraseMatcher # We import the class
import pandas as pd
import time

# 1. Import the library correctly
try:
    from skillNer.skill_extractor_class import SkillExtractor
    from skillNer.general_params import SKILL_DB
    print("✅ 'skillNer' library imported successfully.")
except ImportError as e:
    print("❌ ERROR: The library could not be imported.")
    raise e

print("IP 1 - Step 4 started: loading models...")

# 2. Load the spaCy model
try:
    nlp = spacy.load("en_core_web_lg")
except IOError:
    print("ERROR: 'en_core_web_lg' model not found.")
    print("!python -m spacy download en_core_web_lg")
    raise

# 3. Initialize the SkillExtractor (DEFINITIVE FIX)
try:
    # FIX HERE:
    # Instead of 'PhraseMatcher(nlp.vocab)' we pass only 'PhraseMatcher' (the class itself).
    skill_extractor = SkillExtractor(nlp, SKILL_DB, PhraseMatcher)
    print("✅ SkillExtractor initialized successfully.")

except Exception as e:
    print(f"Error while initializing SkillExtractor: {e}")
    raise

# -------------------------------------------------------------------------
# STEP 4.2: EXTRACT SKILLS FROM THE CANDIDATE'S CV
# -------------------------------------------------------------------------
print("\nStep 4.2: Processing the candidate CV text...")

try:
    with open("IP1_output_candidate_full_text.txt", "r", encoding="utf-8") as f:
        cv_full_text = f.read()
except FileNotFoundError:
    print("ERROR: File not found. Check the previous steps.")
    raise

try:
    # Extract skills with SkillNER
    annotations = skill_extractor.annotate(cv_full_text)
    
    cv_skills_list = []
    
    # Full matches
    for match in annotations['results']['full_matches']:
        cv_skills_list.append(match['doc_node_value'])
        
    # N-gram matches
    for match in annotations['results']['ngram_scored']:
        cv_skills_list.append(match['doc_node_value'])
        
    # Deduplicate and clean
    cv_skills_list = list(set([s.lower() for s in cv_skills_list]))
    
    print(f"-> {len(cv_skills_list)} skills extracted from the CV.")
    print(f"Examples: {cv_skills_list[:5]}")
    
    # Save
    pd.DataFrame(cv_skills_list, columns=["extracted_skill_name"]).to_pickle("IP1_output_candidate_extracted_skills.pkl")

except Exception as e:
    print(f"An error occurred while processing the CV: {e}")

# -------------------------------------------------------------------------
# STEP 4.3: EXTRACT SKILLS FROM THE PROJECT LISTINGS
# -------------------------------------------------------------------------
print("\nStep 4.3: Processing the project listings...")

try:
    df_projects = pd.read_csv("youth_portal_listings_detailed.csv")
    df_projects = df_projects.dropna(subset=['full_description'])
except FileNotFoundError:
    raise

project_skill_list = []
start_time = time.time()

# Counter so that it works like a progress bar
count = 0
total = len(df_projects)

for text in df_projects["full_description"].astype(str):
    try:
        ann = skill_extractor.annotate(text)
        skills = []
        for match in ann['results']['full_matches']:
            skills.append(match['doc_node_value'])
        for match in ann['results']['ngram_scored']:
            skills.append(match['doc_node_value'])
        
        project_skill_list.append(list(set([s.lower() for s in skills])))
    except:
        project_skill_list.append([])
    
    count += 1
    if count % 50 == 0:
        print(f"   ...{count}/{total} listings processed.")

print(f"-> {total} listings processed in {time.time() - start_time:.2f} seconds.")

df_projects["extracted_skills"] = project_skill_list
df_projects.to_pickle("IP1_output_listings_with_skills.pkl")

print("\n--- IP 1 - Step 4 COMPLETED ---")
print(df_projects[["link", "extracted_skills"]].head())

✅ 'skillNer' library imported successfully.
IP 1 - Step 4 started: loading models...
loading full_matcher ...
loading abv_matcher ...
loading full_uni_matcher ...
loading low_form_matcher ...
loading token_matcher ...
✅ SkillExtractor initialized successfully.

Step 4.2: Processing the candidate CV text...
-> 13 skills extracted from the CV.
Examples: ['social medium', 'c', 'html', 'sql server', 'c++']

Step 4.3: Processing the project listings...


/Users/acetiner/miniconda3/envs/tubitak-env/lib/python3.11/site-packages/skillNer/utils.py:99: UserWarning: [W008] Evaluating Token.similarity based on empty vectors.
  vec_similarity = token1.similarity(token2)


   ...50/952 listings processed.
   ...100/952 listings processed.
   ...150/952 listings processed.
   ...200/952 listings processed.
   ...250/952 listings processed.
   ...300/952 listings processed.
   ...350/952 listings processed.
   ...400/952 listings processed.
   ...450/952 listings processed.
   ...500/952 listings processed.
   ...550/952 listings processed.
   ...600/952 listings processed.
   ...650/952 listings processed.
   ...700/952 listings processed.
   ...750/952 listings processed.
   ...800/952 listings processed.
   ...850/952 listings processed.
   ...900/952 listings processed.
   ...950/952 listings processed.
-> 952 listings processed in 1703.07 seconds.

--- IP 1 - Step 4 COMPLETED ---
                                                link  \
0  https://youth.europa.eu/solidarity/opportunity...   
1  https://youth.europa.eu/solidarity/opportunity...   
2  https://youth.europa.eu/solidarity/opportunity...   
3  https://youth.europa.eu/solidarity/opportunity...

In [1]:
!pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 4.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 7.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]


In [2]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import pandas as pd
import time
from tqdm import tqdm


model_name = "xlm-roberta-base"

print("IP 1 - Step 5 started: vectorization with XLM-RoBERTa (ESCO basis)...")

# 1. Device setting
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Load the model (UPDATED)
model_name = "xlm-roberta-base" # A public model that definitely works
print(f"Loading model: {model_name}...")

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()
    print("✅ Model loaded successfully.")
except Exception as e:
    print(f"❌ An error occurred while loading the model: {e}")
    raise e

def get_esco_vectors(text_list):
    """Converts the given skill list into vectors."""
    if not isinstance(text_list, list) or len(text_list) == 0:
        return np.zeros((1, 768)) # The XLM-R-base dimension is 768
        
    with torch.no_grad():
        inputs = tokenizer(text_list, padding=True, truncation=True, return_tensors="pt").to(device)
        outputs = model(**inputs)
        embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
        return embeddings

# -------------------------------------------------------------------------
# STEP 5.1: VECTORIZE THE CANDIDATE'S SKILLS
# -------------------------------------------------------------------------
print("\nStep 5.1: Vectorizing the candidate skills...")

try:
    df_cv_skills = pd.read_pickle("IP1_output_candidate_extracted_skills.pkl")
    cv_skills_list = df_cv_skills["extracted_skill_name"].tolist()

    if cv_skills_list:
        candidate_skill_vectors = get_esco_vectors(cv_skills_list)
        print(f"-> {len(cv_skills_list)} candidate skills vectorized.")
        print(f"Vector matrix shape: {candidate_skill_vectors.shape}")
        np.save("IP1_output_candidate_vectors.npy", candidate_skill_vectors)
    else:
        print("WARNING: The candidate skill list is empty!")
except FileNotFoundError:
    print("ERROR: Candidate skill file not found.")

# -------------------------------------------------------------------------
# STEP 5.2: VECTORIZE THE PROJECT SKILLS
# -------------------------------------------------------------------------
print("\nStep 5.2: Vectorizing the project listings...")

try:
    df_projects = pd.read_pickle("IP1_output_listings_with_skills.pkl")
except FileNotFoundError:
    print("ERROR: Project file not found.")
    raise

start_time = time.time()
df_projects["skill_vectors"] = df_projects["extracted_skills"].apply(get_esco_vectors)
end_time = time.time()

print(f"-> {len(df_projects)} projects vectorized in {end_time - start_time:.2f} seconds.")

# Show the results (with the link)
print(df_projects[["link", "extracted_skills", "skill_vectors"]].head())

# SAVE
df_projects.to_pickle("IP1_output_listings_vectorized.pkl")

print("\n✅✅✅ IP 1 (DATA COLLECTION AND STANDARDIZATION) COMPLETED! ✅✅✅")

IP 1 - Step 5 started: vectorization with XLM-RoBERTa (ESCO basis)...
Using device: mps
Loading model: xlm-roberta-base...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

✅ Model loaded successfully.

Step 5.1: Vectorizing the candidate skills...
-> 13 candidate skills vectorized.
Vector matrix shape: (13, 768)

Step 5.2: Vectorizing the project listings...
-> 952 projects vectorized in 67.58 seconds.
                                                link  \
0  https://youth.europa.eu/solidarity/opportunity...   
1  https://youth.europa.eu/solidarity/opportunity...   
2  https://youth.europa.eu/solidarity/opportunity...   
3  https://youth.europa.eu/solidarity/opportunity...   
4  https://youth.europa.eu/solidarity/opportunity...   

                                 extracted_skills  \
0  [com, teamwork, creative, hosting, food, creat...   
1  [psychological, teamwork, food, learning envir...   
2  [information system, cataloguing, geology envi...   
3  [research, food, data collection, reproductive...   
4  [english, digitalization, food, creativity, di...   

                                   skill_vectors  
0  [[0.016297929, 0.0673945, 0.025580063, 0.